# TODOs
- maps split have data at lon > 200deg


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn import model_selection, metrics
from glob import glob
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import catboost as cb

import sys
from pathlib import Path

current_path = Path.cwd()

parent_path = current_path.parent

sys.path.append(f"{parent_path}/src/highres_ta")

import preprocessing_tools as ppt
import modelling_tools as mot

plt.style.use('default')

# Imports

In [ ]:
# df = ppt.online_GLODAP_import

df = ppt.offline_GLODAP_import(parent_path)

salinity_bin_edges = [0, 32, 34, 36, np.inf]
salinity_bin_labels = ['<32', '32-34', '34-36', '>36']
df['salinity_bin'] = pd.cut(df['salinity_gp'], bins=salinity_bin_edges, labels=salinity_bin_labels)
df['talk_gp_normalized'] = 35*df['talk_gp']/df['salinity_gp']

df = df.dropna(subset=['salinity_bin'])

df.salinity_gp.describe()

# Train-test split

In [ ]:
train_split_df, heldout_test_df = ppt.traintest_salinity_expocode_split(df)

y_train = train_split_df['talk_gp']
x_train = ppt.keep_predictors(train_split_df)

y_test = heldout_test_df['talk_gp']
x_test = ppt.keep_predictors(heldout_test_df)

In [ ]:
cv_folds_dict = ppt.cv_salinity_expocode_split(train_split_df, n_splits=5, plot_cv_splits=True)

In [ ]:
normalized_cv_folds_dict = ppt.cv_salinity_expocode_split(train_split_df, n_splits=5, normalized_y=True, plot_cv_splits=True)

In [ ]:
# # save train-test split to files

# x_train.to_parquet("X_train.parquet")
# x_test.to_parquet("X_test.parquet")
# y_train.to_parquet("y_train.parquet")
# y_test.to_parquet("y_test.parquet")

# Models intercomparison


In [ ]:
import catboost as cb
from sklearn.ensemble import RandomForestRegressor

In [ ]:
train_dir = parent_path / f"trained_models"
rf_dir = train_dir/f"RandomForest"
gp_dir = train_dir/f"GaussianProcesses"
cb_dir = train_dir/f"Catboost"

In [ ]:
test_experiment = mot.Experiment(cv_folds_dict, train_dir, 'not_normalized')
test_experiment.add_a_new_model(regressor = RandomForestRegressor(n_estimators=5, random_state=42), model_name = 'SimpleRandomForest')
test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSE',verbose=0), model_name = 'SimpleCatBoost_RMSE')
test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSEWithUncertainty',verbose=0), model_name = 'SimpleCatBoost_RMSEUncert')
test_experiment.run_full_cv_assessment()


In [ ]:
normalized_test_experiment = mot.Experiment(normalized_cv_folds_dict, train_dir, 'normalized')
normalized_test_experiment.add_a_new_model(regressor = RandomForestRegressor(n_estimators=5, random_state=42), model_name = 'SimpleRandomForest')
normalized_test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSE',verbose=0), model_name = 'SimpleCatBoost_RMSE')
normalized_test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSEWithUncertainty',verbose=0), model_name = 'SimpleCatBoost_RMSEUncert')
normalized_test_experiment.run_full_cv_assessment()

In [ ]:
# model_dir.mkdir(exist_ok=True)
# rf_dir.mkdir(exist_ok=True)
# cb_dir.mkdir(exist_ok=True)
# gp_dir.mkdir(exist_ok=True)

huber score

In [ ]:
# from sklearn.model_selection import learning_curve


# def plot_learning_curve(model, X, y):

#     train_sizes, train_scores, val_scores = learning_curve(
#         model,
#         X,
#         y,
#         cv=5,
#         scoring="neg_root_mean_squared_error"
#     )

#     train_mean = -train_scores.mean(axis=1)
#     val_mean = -val_scores.mean(axis=1)

#     plt.figure(figsize=(7,5))

#     plt.plot(train_sizes, train_mean, label="Train")
#     plt.plot(train_sizes, val_mean, label="Validation")

#     plt.xlabel("Training Set Size")
#     plt.ylabel("RMSE")

#     plt.title("Learning Curve")
#     plt.legend()

#     plt.show()
    
# plot_learning_curve(model_RandomForest, x_train, y_train)

In [ ]:
# def plot_feature_importance(feature_names, importance):

#     import seaborn as sns
    
#     df = pd.DataFrame({
#         "feature": feature_names,
#         "importance": importance
#     })

#     df = df.sort_values("importance", ascending=False)

#     plt.figure(figsize=(7,6))


#     sns.barplot(
#         data=df.head(20),
#         x="importance",
#         y="feature"
#     )

#     plt.title("Feature Importance (Top 20)")

#     plt.show()
    


In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# import modelling_tools as mot
# import shutil

# folder_to_delete = parent_path / f"trained_models/SimpleRandomForest_crossval"
# shutil.rmtree(folder_to_delete)

# test_config = {
#     'regressor' : RandomForestRegressor(n_estimators=5, random_state=42),
#     'model_name': 'SimpleRandomForest_crossval',
#     'parent_dir': parent_path/f"trained_models"
#     }
    

# test_wrapper = mot.ModelWrapper(**test_config)


# # x_train = ppt.keep_predictors(train_split_df)
# # y_train = train_split_df['talk_gp']
# # x_eval = ppt.keep_predictors(heldout_test_df)
# # y_eval = heldout_test_df['talk_gp']

# test_wrapper.cross_validation_run(cv_folds_dict, export_fold_metadata=False)

# Model Intercomparison v2

Catboost

In [ ]:
import catboost as cb

In [ ]:
model = cb.CatBoostRegressor(iterations=2_000, loss_function="RMSEWithUncertainty")
model = model.fit(x_train, y_train, plot=True)

yhat_train, yhat_train_std = model.predict(x_train).T
yhat_test, yhat_test_std = model.predict(x_test).T

In [ ]:
df_metrics = pd.DataFrame({
    'r2_score': {
        'train': metrics.r2_score(y_train, yhat_train), 
        'test': metrics.r2_score(y_test, yhat_test)
    },
    'rmse': {
        'train': metrics.root_mean_squared_error(y_train, yhat_train), 
        'test': metrics.root_mean_squared_error(y_test, yhat_test)
    },
    'mae': {
        'train': metrics.mean_absolute_error(y_train, yhat_train), 
        'test': metrics.mean_absolute_error(y_test, yhat_test)
    },
    'bias': {
        'train': (yhat_train - y_train).mean(), 
        'test': (yhat_test - y_test).mean()
    }
})

df_metrics.T.round(2)

In [ ]:
names: list[str] = model.feature_names_  # type: ignore
values = model.feature_importances_

pd.Series({k: float(v) for k, v in zip(names, values, strict=True)}).sort_values(ascending=False)

RandomForest


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn import model_selection
from sklearn import metrics

In [ ]:
PARAMETERS = {
    "n_estimators": [100],
    "max_depth": [10, None],
    "min_samples_split": [5, 10],
}


parameters = PARAMETERS
model = RandomForestRegressor(n_jobs=-1)

grid_search = model_selection.GridSearchCV(
    estimator=model, param_grid=parameters, cv=5, scoring="neg_mean_squared_error", verbose=2
)

grid_search.fit(
    x_train,
    y_train,
)

In [ ]:
yhat_train = grid_search.best_estimator_.predict(x_train)
yhat_test = grid_search.best_estimator_.predict(x_test)

In [ ]:
df_metrics = pd.DataFrame(
    {
        "r2_score": {
            "train": metrics.r2_score(y_train, yhat_train),
            "test": metrics.r2_score(y_test, yhat_test),
        },
        "rmse": {
            "train": metrics.root_mean_squared_error(y_train, yhat_train),
            "test": metrics.root_mean_squared_error(y_test, yhat_test),
        },
        "mae": {
            "train": metrics.mean_absolute_error(y_train, yhat_train),
            "test": metrics.mean_absolute_error(y_test, yhat_test),
        },
        "bias": {"train": (yhat_train - y_train).mean(), "test": (yhat_test - y_test).mean()},
    }
)

df_metrics.T.round(2)
